# GlobalCLIP -- 02a: Train Standard CombiLayer (no QLayer)

Trains `GlobalCLIPStandardModel`: a frozen PARNET backbone + `MixCoeffHead`
(sequence-dependent mixing coefficients) + per-protein `log_scale`.

**Architecture:**
```
RNA sequence (4×600)
    │
    ▼
PARNET backbone  [frozen]
    │              │
    ▼              ▼
(B,512,L)      (B,223,L) RBP log-prob tracks
    │              │
    ▼              │
MixCoeffHead  ─────┤  alpha (B,223)  [sigmoid]
log_scale     ─────┤  scale (223,)   [exp]
                   ▼
         Weighted sum → (B,1,L) GlobalCLIP prediction
```

**Training target:** log(1+signal) − log(1+control)

**Loss:** Pearson (main) + Multinomial NLL + alpha sparsity


## Set-up

### Imports

In [1]:
import pylbsr.notebooks
import pylbsr.misc

import sys
import torch
import yaml
import json
import pandas as pd
import matplotlib.pyplot as plt
import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from pathlib import Path
from dotmap import DotMap

from parnet_additional_utils import (
    load_parnet_model,
    ParnetModelName,
)
from globalclip_utils import (
    GlobalCLIPStandardModel,
    GlobalCLIPLightningModule,
    GlobalCLIPDataset,
    save_run_config,
)


/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/gin/config.py:615: FutureWarning: `NLLLoss2d` has been deprecated. Please use `NLLLoss` instead as a drop-in replacement and see https://pytorch.org/docs/main/nn.html#torch.nn.NLLLoss for more details.
  decorated_class = decorating_meta(cls.__name__, (cls,), overrides)
Seed set to 42


### Initialisation

In [2]:
_notebook_name = "02a_train_standard.py.ipynb"
_notebook_path = f"notebooks/globalclip/{_notebook_name}"

pylbsr.notebooks.enable_cell_timing_metadata(show=True)
logger = pylbsr.misc.init_logger(_notebook_name)
PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")


[09:41:04] INFO - Project directory: /mnt/storage1/workspace/pgoldemund/parnet--idea1-reconstruction-head


### Parameters

In [10]:
params_gpu_index          = 0
params_run_id             = "globalclip.standard.v3"

# Dataset
params_control_dataset    = "globalclip_lysate_noNHS"   # recommended (cleanest background)
params_seq_length         = 600
params_batch_size         = 64
params_num_rbps           = 223

# Model
params_mix_hidden         = 128    # hidden size of the MixCoeffHead MLP

# Training
params_lr                 = 1e-4
params_max_epochs         = 50
params_lambda_nll         = 0.1    # weight on Multinomial NLL term
params_lambda_alpha       = 0.1    # sparsity penalty on mixing coefficients
params_early_stop_patience= 8      # stop if val/loss doesn't improve
params_num_workers        = 4


⏱ 0.00 s (00:00:00)


### Filepaths and device

In [4]:
pylbsr.misc.set_seed(42)

device = torch.device(f"cuda:{params_gpu_index}" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(params_gpu_index)
    logger.info(f"GPU: {torch.cuda.get_device_name(device)}")
else:
    logger.warning("No GPU available — running on CPU (will be slow).")

_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.server.yaml").read_text())
pretrained_model_name = ParnetModelName.PARNET_7M_0_0

def _res(p):
    p = Path(p)
    return p if p.is_absolute() else PROJECT_DIR / p

FILEPATHS = DotMap()
FILEPATHS.pretrained_model = _res(_fp_cfg["models"][pretrained_model_name.value])
FILEPATHS.dataset          = _res(_fp_cfg["data"][params_control_dataset]["pt"])
FILEPATHS.rbp_names        = PROJECT_DIR / "results" / "globalclip" / "datasets" / "rbp_names.txt"
FILEPATHS.output_dir       = PROJECT_DIR / _fp_cfg["results"]["standard_model"] / params_run_id
FILEPATHS.output_dir.mkdir(parents=True, exist_ok=True)

for k, v in FILEPATHS.items():
    logger.info(f"{k:25s}: {v}")


Seed set to 42


[09:41:04] INFO - GPU: NVIDIA RTX A4000
[09:41:04] INFO - pretrained_model         : /mnt/storage1/ml4rg26-shared/parnet-eclip/models-full-rbp-set/parnet.7m-0.0.pt
[09:41:04] INFO - dataset                  : /mnt/storage1/ml4rg26-deconvgclip/provided_data/600nt_globalCLIP_synchronized_datasets/globalclip_lysate_noNHS_600bp_signalfiltered.pt.gz
[09:41:04] INFO - rbp_names                : /mnt/storage1/workspace/pgoldemund/parnet--idea1-reconstruction-head/results/globalclip/datasets/rbp_names.txt
[09:41:04] INFO - output_dir               : /mnt/storage1/workspace/pgoldemund/parnet--idea1-reconstruction-head/results/globalclip/standard/globalclip.standard.v2


⏱ 0.15 s (00:00:00)


## Load data

In [5]:
train_ds = GlobalCLIPDataset(FILEPATHS.dataset, split="train",
                              seq_len=params_seq_length, total_key="globalCLIP")
val_ds   = GlobalCLIPDataset(FILEPATHS.dataset, split="valid",
                              seq_len=params_seq_length, total_key="globalCLIP")

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=params_batch_size, shuffle=True,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=params_batch_size, shuffle=False,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)

logger.info(f"Train: {len(train_ds)} samples  ({len(train_loader)} batches)")
logger.info(f"Valid: {len(val_ds)} samples")

# Quick shape check
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"  batch['{k}']: {tuple(v.shape)}")


Loading (gz) globalclip_lysate_noNHS_600bp_signalfiltered.pt.gz split='train'... loaded 39052 samples.
Loading (gz) globalclip_lysate_noNHS_600bp_signalfiltered.pt.gz split='valid'... loaded 7361 samples.


[09:41:25] INFO - Train: 39052 samples  (611 batches)
[09:41:25] INFO - Valid: 7361 samples


  batch['sequence']: (64, 4, 600)
  batch['signal']: (64, 1, 600)
⏱ 21.68 s (00:00:21)


## Build model

In [6]:
logger.info(f"Loading pretrained PARNET from {FILEPATHS.pretrained_model}")
parnet = load_parnet_model(
    pretrained_model_name,
    FILEPATHS.pretrained_model,
    dtype=torch.float32,
    device=device,
)
# We keep the full 223-task head -- do NOT reset it
parnet.eval()
logger.info("PARNET loaded (223-task head kept frozen).")


[09:41:26] INFO - Loading pretrained PARNET from /mnt/storage1/ml4rg26-shared/parnet-eclip/models-full-rbp-set/parnet.7m-0.0.pt
[09:41:26] INFO - PARNET loaded (223-task head kept frozen).


⏱ 0.33 s (00:00:00)


In [7]:
model = GlobalCLIPStandardModel(
    parnet_model=parnet,
    num_rbps=params_num_rbps,
    mix_hidden=params_mix_hidden,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
logger.info(f"Parameters: {trainable:,} trainable / {total:,} total")

# Sanity check forward pass
with torch.no_grad():
    pred, alpha = model(batch["sequence"].to(device))
print(f"pred shape : {tuple(pred.shape)}   (should be (B, 1, 600))")
print(f"alpha shape: {tuple(alpha.shape)}  (should be (B, 223))")
print(f"alpha range: [{alpha.min():.3f}, {alpha.max():.3f}]")


[09:41:26] INFO - Parameters: 94,654 trainable / 7,629,469 total
/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


pred shape : (64, 1, 600)   (should be (B, 1, 600))
alpha shape: (64, 223)  (should be (B, 223))
alpha range: [0.381, 0.618]
⏱ 0.31 s (00:00:00)


## Train

In [8]:
# Inspiziere die PARNET head-Struktur
print(type(parnet.head))
print(type(parnet.head.head_target))
print(parnet.head.head_target)
# Zeige alle Submodule
for name, module in parnet.named_modules():
    if "head" in name.lower() or "223" in str(module):
        print(f"{name}: {type(module).__name__}")

<class 'parnet.layers.NewAdditiveMix'>
<class 'parnet.layers.LinearProjection'>
LinearProjection(
  (pointwise_conv): Conv1d(512, 223, kernel_size=(1,), stride=(1,), padding=same, bias=False)
)
: RBPNet
head: NewAdditiveMix
head.head_target: LinearProjection
head.head_target.pointwise_conv: Conv1d
head.head_control: LinearProjection
head.head_control.pointwise_conv: Conv1d
head.mix_coeff: NewMixCoeffMLP
head.mix_coeff.dense1: Linear
head.mix_coeff.act: GELU
head.mix_coeff.dense2: Linear
head.penalty: NewMixCoeffPenalty
⏱ 0.00 s (00:00:00)


In [9]:
lightning_model = GlobalCLIPLightningModule(
    model=model,
    lr=params_lr,
    lambda_nll=params_lambda_nll,
    lambda_alpha=params_lambda_alpha,
)

callbacks = [
    ModelCheckpoint(
        dirpath=FILEPATHS.output_dir / "checkpoints",
        filename="best-{epoch:02d}-{val/loss:.4f}",
        monitor="val/loss",
        mode="min",
        save_top_k=2,
    ),
    EarlyStopping(
        monitor="val/loss",
        patience=params_early_stop_patience,
        mode="min",
    ),
]

loggers = [
    CSVLogger(str(FILEPATHS.output_dir), name="csv_logs"),
    TensorBoardLogger(str(FILEPATHS.output_dir), name="tb_logs"),
]

trainer = pl.Trainer(
    max_epochs=params_max_epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=[params_gpu_index] if torch.cuda.is_available() else 1,
    callbacks=callbacks,
    logger=loggers,
    log_every_n_steps=50,
    deterministic=True,
)

logger.info("Starting training...")
trainer.fit(lightning_model, train_loader, val_loader)
logger.info("Training complete.")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
[09:41:26] INFO - Starting training...
2026-07-01 09:41:27.053819: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-01 09:41:27.066665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-01 09:41:27.085875: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-01 09:41:27.085910: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for pl

Sanity Checking: |                                            | 0/? [00:00<?, ?it/s]

/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:527: Found 62 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |                                                   | 0/? [00:00<?, ?it/s]

Validation: |                                                 | 0/? [00:00<?, ?it/s]

...

Validation: |                                                 | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

⏱ 965.01 s (00:16:05)


/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Save model and run config

In [ ]:
# Save full model (state dict only -- can be reloaded with GlobalCLIPStandardModel)
torch.save(model.state_dict(), FILEPATHS.output_dir / "model.statedict.pt")
# Also save the full object for quick prototyping
torch.save(model, FILEPATHS.output_dir / "model.full.pt")

run_cfg = {
    "model_type":           "GlobalCLIPStandardModel",
    "pretrained_model_name": pretrained_model_name.value,
    "control_dataset":       params_control_dataset,
    "params_seq_length":     params_seq_length,
    "params_batch_size":     params_batch_size,
    "params_num_rbps":       params_num_rbps,
    "params_mix_hidden":     params_mix_hidden,
    "params_lr":             params_lr,
    "params_max_epochs":     params_max_epochs,
    "params_lambda_nll":     params_lambda_nll,
    "params_lambda_alpha":   params_lambda_alpha,
    "dataset_path":          str(FILEPATHS.dataset),
    "output_dir":            str(FILEPATHS.output_dir),
}
save_run_config(FILEPATHS.output_dir, run_cfg)
logger.info(f"Model and config saved to {FILEPATHS.output_dir}")


## Training curves

In [ ]:
csv_log_dir = FILEPATHS.output_dir / "csv_logs"
metrics_paths = sorted(csv_log_dir.glob("version_*/metrics.csv"))
assert metrics_paths, f"No CSV log found under {csv_log_dir}"
df_csv = pd.read_csv(metrics_paths[-1])
epoch_df = df_csv.groupby("epoch").last().reset_index()

plots = [
    ({"train": "train/loss_epoch", "val": "val/loss"}, "Total loss"),
    ({"train": "train/pearson_epoch", "val": "val/pearson"}, "Pearson loss"),
    ({"val alpha_mean": "val/alpha_mean_epoch"}, "Mean alpha (sparsity)"),
]
fig, axes = plt.subplots(1, len(plots), figsize=(14, 4))
for (col_dict, title), ax in zip(plots, axes):
    for label, col in col_dict.items():
        if col in epoch_df.columns:
            epoch_df.plot("epoch", col, ax=ax, label=label, marker="o", markersize=3)
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.suptitle(f"Training metrics -- {params_run_id}")
plt.tight_layout()
plt.savefig(FILEPATHS.output_dir / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()


## Quick look at learned mixing coefficients

In [ ]:
rbp_names = (PROJECT_DIR / "results" / "globalclip" / "datasets" / "rbp_names.txt").read_text().strip().split("\n")

model.eval()
with torch.no_grad():
    pred_ex, alpha_ex = model(batch["sequence"].to(device))

mean_alpha_batch = alpha_ex.mean(0).cpu().numpy()
top_idx = mean_alpha_batch.argsort()[::-1][:20]

print("Top-20 proteins by mixing coefficient (this batch):")
for rank, i in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {rbp_names[i]:<20s}  alpha={mean_alpha_batch[i]:.4f}")

# log_scale: positive = upweighted, negative = downweighted
log_scale_vals = model.log_scale.detach().cpu().numpy()
top_scale_idx = log_scale_vals.argsort()[::-1][:10]
print("\nTop-10 proteins by exp(log_scale):")
for i in top_scale_idx:
    import math
    print(f"  {rbp_names[i]:<20s}  exp(log_scale)={math.exp(log_scale_vals[i]):.4f}")


## Next steps

- Run `03_evaluate_and_analyze.py.ipynb` for full test-set evaluation
- Run `02b_train_qlayer.py.ipynb` to train the QLayer model and compare
- The saved model is at: `results/globalclip/standard/{params_run_id}/model.statedict.pt`
